# Boosting to Uniformity (hep_ml) applied to the ggH BDT

This notebook adapts the [hep_ml `BoostingToUniformity` demo](https://github.com/arogozhnikov/hep_ml/blob/master/notebooks/BoostingToUniformity.ipynb)
to our own signal/background samples.

The problem `hep_ml` solves is exactly the one investigated in the repo's sculpting study
(see `plot_macros/SculptingStudy.ipynb`): a plain BDT tends to sculpt the background so that a cut
on the discriminator carves a fake peak into `diMuon_bsConstrainedMass`, which can fake a signal-like
bump under the H -> mumu peak. `hep_ml` provides boosting variants (`uBoost`, and `UGradientBoosting`
with a kNN-based flatness loss) that explicitly penalize non-uniform *background* efficiency along a
chosen feature -- here, `diMuon_bsConstrainedMass` -- while still optimizing signal/background
separation on the remaining kinematic variables.

Classifiers compared:
* plain **GradientBoosting** (sklearn) -- our current nominal-style baseline, not mass-uniform.
* **uGB+knnAda** -- gradient boosting with the kNN-Ada loss (`hep_ml.gradientboosting.KnnAdaLossFunction`).
* **uGB+FlatnessLoss** -- gradient boosting with the kNN flatness loss
  (`hep_ml.gradientboosting.KnnFlatnessLossFunction`).
* **uBoost** -- optional, much slower to train (`hep_ml.uboost.uBoostClassifier`); left disabled by
  default below, flip `RUN_UBOOST = True` to include it.

Unlike the original Dalitz-plot demo (which wants **signal** flat in the Dalitz features), we want the
**background** (label 0) to have flat efficiency in `diMuon_bsConstrainedMass`, so `uniform_label=0`
everywhere below.

Requires `hep_ml` in the `xgboost_env` micromamba environment (`pip3 install hep_ml`) -- it is not part
of the base env described in `python/xgboost/README.md` yet.


In [ ]:
import os

import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import mplhep as hep
from sklearn.metrics import auc, roc_auc_score, roc_curve

from hep_ml import gradientboosting as ugb
from hep_ml import uboost
from hep_ml.commonutils import train_test_split
from hep_ml.metrics import BinBasedSDE, KnnBasedCvM
from sklearn.ensemble import GradientBoostingClassifier

import xgb_utils

hep.style.use("CMS")
%matplotlib inline


## Load the same signal/background samples used in `BDTOptimizationTests.ipynb`

In [ ]:
channel_US, era = "ggH", "Combined"
background_subset = "Full"
signal_subset = "NottH"
SIGNAL_REGION = (115, 135)

plot_dir = "BDTplots/uniformity_study/"
os.makedirs(plot_dir, exist_ok=True)

variables = xgb_utils.get_variables(channel_US, True)
train_features = [var[0] for var in variables[:-3]]
mass_var = "diMuon_bsConstrainedMass"
uniform_features = [mass_var]

print("number of training variables:", len(train_features))
for var in train_features:
    print(" ", var)


In [ ]:
data_directory = f"../../root_io/skim/{channel_US}/"
signal_file_name = f"{data_directory}signal_{era}_skim_{signal_subset}.root"
bkg_file_name = f"{data_directory}background_{era}_skim_{background_subset}.root"

print("[INFO]: Creating data frames")
with uproot.open(signal_file_name) as file:
    df_signal = pd.DataFrame(
        file["tree_output"].arrays([var[0] for var in variables], library="np"),
    )
with uproot.open(bkg_file_name) as file:
    df_bkg = pd.DataFrame(
        file["tree_output"].arrays([var[0] for var in variables], library="np"),
    )
print("[INFO]: Data frames created successfully")
print("signal events:", len(df_signal), "| background events:", len(df_bkg))


## Build the labeled dataset

`hep_ml`'s uniform-boosting classifiers take a `pandas.DataFrame` and select columns by name
(`train_features` for the split variables, `uniform_features` for the variable(s) whose efficiency
should stay flat), so -- unlike the xgboost notebooks in this directory -- we keep everything as a
DataFrame instead of converting to a raw numpy matrix.


In [ ]:
signal_weight_sum = df_signal["weight"].sum()
bkg_weight_sum = df_bkg["weight"].sum()
print("Sig weight sum (raw):", signal_weight_sum)
print("Bkg weight sum (raw):", bkg_weight_sum)

# match BDTOptimizationTests.ipynb: renormalize signal weight sum to the background weight sum
df_signal = df_signal.copy()
df_signal["weight"] = df_signal["weight"] * (bkg_weight_sum / signal_weight_sum)

df_signal["label"] = 1
df_bkg["label"] = 0

df_all = pd.concat([df_signal, df_bkg], ignore_index=True)
print("total events:", len(df_all))


In [ ]:
SEED = 7
TEST_SIZE = 0.4

train_df, test_df = train_test_split(df_all, test_size=TEST_SIZE, random_state=SEED)

y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()
w_train = train_df["weight"].to_numpy()
w_test = test_df["weight"].to_numpy()

print(f"train: {len(train_df)} events ({y_train.sum():.0f} signal-weighted)")
print(f"test:  {len(test_df)} events ({y_test.sum():.0f} signal-weighted)")


## Set up classifiers

`n_estimators` is kept modest (this is a uniformity *test*, not a final trained model) since `uGB+FL`
and `uBoost` are considerably slower than plain xgboost.


In [ ]:
RUN_UBOOST = False  # uBoost is slow to train -- flip to True once the quick classifiers look sane

n_estimators = 150
max_depth = 4

classifiers = {}

classifiers["GradientBoosting"] = GradientBoostingClassifier(
    max_depth=max_depth, n_estimators=n_estimators, learning_rate=0.1
)

knnloss = ugb.KnnAdaLossFunction(uniform_features, knn=10, uniform_label=0)
classifiers["uGB+knnAda"] = ugb.UGradientBoostingClassifier(
    loss=knnloss,
    max_depth=max_depth,
    n_estimators=n_estimators,
    learning_rate=0.4,
    train_features=train_features,
)

flatnessloss = ugb.KnnFlatnessLossFunction(
    uniform_features, fl_coefficient=3.0, power=1.3, uniform_label=0
)
classifiers["uGB+FL"] = ugb.UGradientBoostingClassifier(
    loss=flatnessloss,
    max_depth=max_depth,
    n_estimators=n_estimators,
    learning_rate=0.1,
    train_features=train_features,
)

if RUN_UBOOST:
    from sklearn.tree import DecisionTreeClassifier

    classifiers["uBoost"] = uboost.uBoostClassifier(
        uniform_features=uniform_features,
        uniform_label=0,
        base_estimator=DecisionTreeClassifier(max_depth=max_depth),
        n_estimators=n_estimators,
        train_features=train_features,
        efficiency_steps=12,
        n_threads=4,
    )

list(classifiers)


In [ ]:
for name, clf in classifiers.items():
    print(f"Training {name}...")
    if name == "GradientBoosting":
        clf.fit(train_df[train_features], y_train, sample_weight=w_train)
    else:
        # uGB/uBoost classifiers pick train_features/uniform_features out of the full frame themselves
        clf.fit(train_df, y_train, sample_weight=w_train)

print("Training completed!")


## ROC curves

In [ ]:
def get_score(name, clf, df):
    if name == "GradientBoosting":
        return clf.predict_proba(df[train_features])[:, 1]
    return clf.predict_proba(df)[:, 1]


plt.figure(figsize=(8, 8))
for name, clf in classifiers.items():
    y_score = get_score(name, clf, test_df)
    fpr, tpr, _ = roc_curve(y_test, y_score, sample_weight=w_test)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"{channel_US} {era} - ROC")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig(plot_dir + "roc_comparison.png")
plt.savefig(plot_dir + "roc_comparison.pdf")
plt.show()


## Non-uniformity metrics (background flatness in `diMuon_bsConstrainedMass`)

Lower is better for both SDE (squared deviation of efficiency) and CvM (Cramer-von Mises distance) --
these quantify how much the background selection efficiency varies across `diMuon_bsConstrainedMass`
bins/neighbourhoods, i.e. how much each classifier sculpts the background mass spectrum.


In [ ]:
bkg_test_mask = y_test == 0

sde_metric = BinBasedSDE(uniform_features, uniform_label=0)
sde_metric.fit(test_df, y_test, sample_weight=w_test)

cvm_metric = KnnBasedCvM(uniform_features, uniform_label=0)
cvm_metric.fit(test_df, y_test, sample_weight=w_test)

print(f"{'classifier':<20}{'SDE (bkg flatness)':<22}{'CvM (bkg flatness)'}")
for name, clf in classifiers.items():
    proba = get_score(name, clf, test_df)
    proba_2d = np.column_stack([1 - proba, proba])
    sde_value = sde_metric(y_test, proba_2d, sample_weight=w_test)
    cvm_value = cvm_metric(y_test, proba_2d, sample_weight=w_test)
    print(f"{name:<20}{sde_value:<22.6f}{cvm_value:.6f}")


## Sculpting study: background `diMuon_bsConstrainedMass` shape before/after a BDT cut

This is the same style of plot as `plot_macros/SculptingStudy.ipynb`, repeated per classifier: the
background mass spectrum in the signal region should stay flat/unbiased after cutting on the
discriminator. A plain BDT is expected to visibly reshape (sculpt) the spectrum; the uniform-boosting
variants should leave it closer to the pre-cut shape.


In [ ]:
mass_test = test_df.loc[bkg_test_mask, mass_var].to_numpy()
weight_test_bkg = w_test[bkg_test_mask]
mass_bins = np.linspace(SIGNAL_REGION[0], SIGNAL_REGION[1], 21)

bdt_cut_quantile = 0.5  # keep the top 50% of background by discriminator score, per classifier

fig, axs = plt.subplots(1, len(classifiers), figsize=(6 * len(classifiers), 5), sharey=True)
if len(classifiers) == 1:
    axs = [axs]

region_mask = (mass_test > SIGNAL_REGION[0]) & (mass_test < SIGNAL_REGION[1])

counts_before, _ = np.histogram(
    mass_test[region_mask], bins=mass_bins, weights=weight_test_bkg[region_mask]
)
counts_before = counts_before / counts_before.sum()

for ax, (name, clf) in zip(axs, classifiers.items()):
    bkg_score = get_score(name, clf, test_df)[bkg_test_mask]
    cut = np.quantile(bkg_score, bdt_cut_quantile)
    after_mask = region_mask & (bkg_score > cut)

    counts_after, _ = np.histogram(
        mass_test[after_mask], bins=mass_bins, weights=weight_test_bkg[after_mask]
    )
    counts_after = counts_after / counts_after.sum()

    hep.histplot(counts_before, bins=mass_bins, ax=ax, label="before cut")
    hep.histplot(counts_after, bins=mass_bins, ax=ax, label=f"after cut (score > {cut:.2f})")
    ax.set_xlabel(r"$M_{\mu\mu}$ (bsC) [GeV]")
    ax.set_title(name)
    ax.legend()

axs[0].set_ylabel("Normalized background events")
plt.tight_layout()
plt.savefig(plot_dir + "sculpting_comparison.png")
plt.savefig(plot_dir + "sculpting_comparison.pdf")
plt.show()
